In [2]:
from dotenv import load_dotenv
import os

In [3]:
load_dotenv("../.env") 
MONGO_URI = os.getenv("MONGO_URI")
print(MONGO_URI[:10] + "...") 

mongodb+sr...


In [4]:
from pymongo import MongoClient
import pandas as pd
client = MongoClient(MONGO_URI)
db = client["veille_media"]
collection = db["articles"]
data = list(collection.find())
df = pd.DataFrame(data)

df.head()

,_id,id_article,source,titre,date_publication,contenu,url,categorie,sentiment,source_type,created_at,sentiment_score
0,696f6450803a3bd1d5076f9f,https://www.madagascar-tribune.com/Conjoncture...,https://www.madagascar-tribune.com/spip.php?pa...,Richard Ravalomanana et Marie Michelle Sahondr...,"Tue, 20 Jan 2026 08:36:32 +0300",Deux figures emblématiques du régime Rajoelina...,https://www.madagascar-tribune.com/Conjoncture...,politique,neutre,NaN,NaN,NaN
1,696f6450803a3bd1d5076fa0,https://www.madagascar-tribune.com/Senat-la-HC...,https://www.madagascar-tribune.com/spip.php?pa...,Sénat : la HCC confirme la fin du mandat et l'...,"Tue, 20 Jan 2026 08:31:21 +0300",La Haute Cour constitutionnelle (HCC) a offici...,https://www.madagascar-tribune.com/Senat-la-HC...,autre,neutre,NaN,NaN,NaN
2,696f6451803a3bd1d5076fa1,https://www.madagascar-tribune.com/103-signatu...,https://www.madagascar-tribune.com/spip.php?pa...,103 poids plume,"Mon, 19 Jan 2026 08:35:00 +0300",Lors d'« Assises des Partis Politiques » pour ...,https://www.madagascar-tribune.com/103-signatu...,autre,negatif,NaN,NaN,NaN
3,696f6451803a3bd1d5076fa2,https://www.madagascar-tribune.com/Concertatio...,https://www.madagascar-tribune.com/spip.php?pa...,Concertation nationale : le président de la Re...,"Mon, 19 Jan 2026 08:00:00 +0300",Le colonel Randrianarina est venu à la rescous...,https://www.madagascar-tribune.com/Concertatio...,politique,positif,NaN,NaN,NaN
4,696f6451803a3bd1d5076fa3,https://www.madagascar-tribune.com/Administrat...,https://www.madagascar-tribune.com/spip.php?pa...,Herintsalama Rajaonarivelo ferme la porte à un...,"Mon, 19 Jan 2026 07:00:00 +0300","Ce dimanche, devant des milliers de fidèles de...",https://www.madagascar-tribune.com/Administrat...,autre,neutre,NaN,NaN,NaN


In [5]:
print("Total articles:", len(df))
print(df.info())
print(df.isna().sum())

Total articles: 2092
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2092 entries, 0 to 2091
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   _id               2092 non-null   object 
 1   id_article        2092 non-null   object 
 2   source            2092 non-null   object 
 3   titre             2092 non-null   object 
 4   date_publication  2084 non-null   object 
 5   contenu           2092 non-null   object 
 6   url               2092 non-null   object 
 7   categorie         2092 non-null   object 
 8   sentiment         2092 non-null   object 
 9   source_type       1165 non-null   object 
 10  created_at        1165 non-null   object 
 11  sentiment_score   0 non-null      float64
dtypes: float64(1), object(11)
memory usage: 196.3+ KB
None
_id                    0
id_article             0
source                 0
titre                  0
date_publication       8
contenu                0
url 

In [7]:
from collections import Counter
types = Counter()

for doc in collection.find({}, {"date_publication": 1}):
    value = doc.get("date_publication")
    types[type(value).__name__] += 1

print(types)

Counter({'str': 1319, 'datetime': 765, 'NoneType': 8})


In [8]:
sample_dates = collection.find(
    {"date_publication": {"$type": "string"}},
    {"date_publication": 1}
).limit(20)

for doc in sample_dates:
    print(doc["date_publication"])

Tue, 20 Jan 2026 08:36:32 +0300
Tue, 20 Jan 2026 08:31:21 +0300
Mon, 19 Jan 2026 08:35:00 +0300
Mon, 19 Jan 2026 08:00:00 +0300
Mon, 19 Jan 2026 07:00:00 +0300
Sat, 17 Jan 2026 09:36:00 +0300
Sat, 17 Jan 2026 09:35:49 +0300
Sat, 17 Jan 2026 09:35:38 +0300
Fri, 16 Jan 2026 09:35:00 +0300
Fri, 16 Jan 2026 09:32:53 +0300
Fri, 16 Jan 2026 08:50:01 +0300
Thu, 15 Jan 2026 08:38:48 +0300
Thu, 15 Jan 2026 07:00:00 +0300
Thu, 15 Jan 2026 06:00:00 +0300
Wed, 14 Jan 2026 08:25:15 +0300
Wed, 14 Jan 2026 06:00:00 +0300
Tue, 13 Jan 2026 08:33:24 +0300
Tue, 13 Jan 2026 07:00:00 +0300
Mon, 12 Jan 2026 09:40:50 +0300
Mon, 12 Jan 2026 07:00:00 +0300


In [21]:
from dateutil import parser

invalid_docs = []

for doc in collection.find({"date_publication": {"$type": "string"}}):
    try:
        parser.parse(doc["date_publication"])
    except:
        invalid_docs.append(doc["_id"])

print("Dates invalides :", len(invalid_docs))


Dates invalides : 69


In [10]:
sample_dates = collection.find(
    {"date_publication": {"$type": "date"}},
    {"date_publication": 1}
).limit(20)

for doc in sample_dates:
    print(doc["date_publication"])

2026-01-20 13:43:12
2026-01-20 14:57:04
2026-01-20 15:02:56
2026-01-21 05:01:34
2026-01-21 04:53:01
2026-01-21 04:52:36
2026-01-21 04:52:32
2026-01-21 04:52:13
2026-01-21 03:00:00
2026-01-21 03:00:00
2026-01-21 03:00:00
2026-01-21 03:00:00
2026-01-21 03:00:00
2026-01-21 03:00:00
2026-01-21 03:00:00
2026-01-21 03:00:00
2026-01-21 03:00:00
2026-01-21 03:00:00
2026-01-21 07:59:33
2026-01-21 07:46:28


In [11]:
from dateutil import parser
from datetime import datetime

for doc in collection.find({"date_publication": {"$type": "string"}}):
    try:
        new_date = parser.parse(doc["date_publication"])
        
        collection.update_one(
            {"_id": doc["_id"]},
            {"$set": {"date_publication": new_date}}
        )
    except Exception as e:
        print(f"Erreur sur {doc['_id']} : {doc['date_publication']}")


Erreur sur 698b52f9e2237c9990fbc2dc : Date:10/2/2026
Erreur sur 698b52f9e2237c9990fbc2dd : Date:10/2/2026
Erreur sur 698b52f9e2237c9990fbc2de : Date:10/2/2026
Erreur sur 698b52fae2237c9990fbc2df : Date:10/2/2026
Erreur sur 698b52fae2237c9990fbc2e0 : Date:10/2/2026
Erreur sur 698b52fae2237c9990fbc2e1 : Date:10/2/2026
Erreur sur 698b52fae2237c9990fbc2e2 : Date:10/2/2026
Erreur sur 698b52fae2237c9990fbc2e3 : Date:10/2/2026
Erreur sur 698b52fae2237c9990fbc2e4 : Date:10/2/2026
Erreur sur 698b52fbe2237c9990fbc2e5 : Date:10/2/2026
Erreur sur 698b52fbe2237c9990fbc2e6 : Date:9/2/2026
Erreur sur 698b52fbe2237c9990fbc2e7 : Date:9/2/2026
Erreur sur 698b52fbe2237c9990fbc2e8 : Date:9/2/2026
Erreur sur 698b52fbe2237c9990fbc2e9 : Date:9/2/2026
Erreur sur 698b52fbe2237c9990fbc2ea : Date:9/2/2026
Erreur sur 698b52fbe2237c9990fbc2eb : Date:9/2/2026
Erreur sur 698b52fce2237c9990fbc2ec : Date:9/2/2026
Erreur sur 698b52fce2237c9990fbc2ed : Date:9/2/2026
Erreur sur 698b52fce2237c9990fbc2ee : Date:9/2/2026
Er

In [12]:
from datetime import datetime

def normalize_date(date_str):
    if not date_str:
        return None
    
    cleaned = date_str.replace("Date:", "").strip()
    
    try:
        return datetime.strptime(cleaned, "%d/%m/%Y")
    except:
        return None


for doc in collection.find():
    
    date_pub = doc.get("date_publication")
    created = doc.get("created_at")
    
    new_date = None
    
    # 1️⃣ Si déjà datetime
    if isinstance(date_pub, datetime):
        continue
    
    # 2️⃣ Si string → essayer conversion
    if isinstance(date_pub, str):
        new_date = normalize_date(date_pub)
    
    # 3️⃣ Si vide ou conversion échoue → fallback created_at
    if not new_date:
        if isinstance(created, datetime):
            new_date = created
    
    # 4️⃣ Update si on a trouvé une date
    if new_date:
        collection.update_one(
            {"_id": doc["_id"]},
            {"$set": {"date_publication": new_date}}
        )
    else:
        print(f"Aucune date valide pour {doc['_id']}")


Aucune date valide pour 696f91b6c613d99e9417630f
Aucune date valide pour 696f91b8c613d99e94176310
Aucune date valide pour 696f91b9c613d99e94176311
Aucune date valide pour 696f91bbc613d99e94176312
Aucune date valide pour 696f91bcc613d99e94176313
Aucune date valide pour 696f91bec613d99e94176314
Aucune date valide pour 696f91bfc613d99e94176315
Aucune date valide pour 6970a359bf9f87597b924b1e
Aucune date valide pour 698b52fce2237c9990fbc2ef


In [13]:
doc = collection.find_one({"_id": "696f91b6c613d99e9417630f"})
print(doc)


None


In [14]:
from bson import ObjectId

doc = collection.find_one({"_id": ObjectId("696f91b6c613d99e9417630f")})
print(doc)


{'_id': ObjectId('696f91b6c613d99e9417630f'), 'id_article': 'https://www.malagasynews.com/actualites/diplomatie-sa-majeste-le-roi-repond-favorablement-a-linvitation-du-president-donald-j-trump-pour-devenir-membre-fondateur-du-conseil-de-paix-mae/', 'source': 'Malagasy News', 'titre': 'Diplomatie: Sa Majesté le Roi répond favorablement à l’invitation du Président Donald J Trump pour devenir Membre Fondateur du Conseil de Paix (MAE)', 'date_publication': None, 'contenu': 'La participation à ce Conseil est réservée à un groupe restreint de leaders de stature internationale, engagés en faveur d’un avenir sûr et prospère pour les générations futures, précise le communiqué, ajoutant que cette invitation constitue une reconnaissance du leadership éclairé de Sa Majesté le Roi, et de Sa Stature en tant qu’acteur de paix incontournable. Elle témoigne de la confiance dont jouit le Souverain auprès du Président des États-Unis et de la communauté internationale. Tout en saluant l’engagement et la v

In [15]:
from bson import ObjectId

for doc in collection.find({
    "date_publication": None
}):
    
    object_date = doc["_id"].generation_time
    
    collection.update_one(
        {"_id": doc["_id"]},
        {"$set": {"date_publication": object_date}}
    )


In [17]:
collection.aggregate([
    {"$group": {"_id": {"$type": "$date_publication"}, "count": {"$sum": 1}}}
])


In [18]:
result = collection.aggregate([
    {"$group": {"_id": {"$type": "$date_publication"}, "count": {"$sum": 1}}}
])

for r in result:
    print(r)


{'_id': 'date', 'count': 2091}
{'_id': 'string', 'count': 1}


In [19]:
doc = collection.find_one(
    {"date_publication": {"$type": "string"}},
    {"date_publication": 1}
)

print(doc)


{'_id': ObjectId('698b52fce2237c9990fbc2ef'), 'date_publication': ''}


In [20]:
from bson import ObjectId

doc_id = ObjectId("698b52fce2237c9990fbc2ef")

collection.update_one(
    {"_id": doc_id},
    {"$set": {"date_publication": doc_id.generation_time}}
)


UpdateResult({'n': 1, 'electionId': ObjectId('7fffffff000000000000003c'), 'opTime': {'ts': Timestamp(1771421272, 1), 't': 60}, 'nModified': 1, 'ok': 1.0, '$clusterTime': {'clusterTime': Timestamp(1771421272, 1), 'signature': {'hash': b'\xa2\xbeN\x95\xdf\xe9l\xc0\x02[\xa6e\xfc/)\xff\xb6\x83\x02\xf3', 'keyId': 7573748671250956292}}, 'operationTime': Timestamp(1771421272, 1), 'updatedExisting': True}, acknowledged=True)

In [21]:
result = collection.aggregate([
    {"$group": {"_id": {"$type": "$date_publication"}, "count": {"$sum": 1}}}
])

for r in result:
    print(r)


{'_id': 'date', 'count': 2092}


In [22]:
collection.create_index("date_publication")


'date_publication_1'

In [23]:
collection.index_information()


{'_id_': {'v': 2, 'key': [('_id', 1)]},
 'date_publication_1': {'v': 2, 'key': [('date_publication', 1)]}}

In [47]:
import pandas as pd
from openpyxl import Workbook
from bson import ObjectId

# 🔹 Récupérer toutes les données
docs = list(collection.find())

# 🔹 Convertir ObjectId en string et gérer les listes
for d in docs:
    d["_id"] = str(d["_id"])
    for key, value in d.items():
        # Si la valeur est une liste, la convertir en chaîne séparée par ", "
        if isinstance(value, list):
            d[key] = ", ".join(str(v) for v in value)

# 🔹 Convertir en DataFrame
df = pd.DataFrame(docs)

# =========================
# 1️⃣ Export CSV
# =========================
csv_path = "export_verification.csv"
df.to_csv(csv_path, index=False)
print("CSV exporté :", csv_path)

# =========================
# 2️⃣ Export Excel
# =========================
xlsx_path = "export_verification.xlsx"

wb = Workbook()
ws = wb.active
ws.title = "Mongo_Export"

# Header
ws.append(list(df.columns))

# Lignes
for row in df.itertuples(index=False):
    ws.append(list(row))

wb.save(xlsx_path)
print("Excel exporté :", xlsx_path)


CSV exporté : export_verification.csv
Excel exporté : export_verification.xlsx


In [25]:
collection.delete_many({
    "$or": [
        {"contenu": ""},
        {"contenu": None},
        {"contenu": {"$regex": r"^\s*$"}}
    ]
})


DeleteResult({'n': 101, 'electionId': ObjectId('7fffffff000000000000003c'), 'opTime': {'ts': Timestamp(1771422537, 11), 't': 60}, 'ok': 1.0, '$clusterTime': {'clusterTime': Timestamp(1771422537, 11), 'signature': {'hash': b'\xe3\xb1\x06\x8b\xdb\x85\xad\x95\x0b\x8e\x00\xa1Z\xb3\x9a\xc1\x88\xa3\xe3\x15', 'keyId': 7573748671250956292}}, 'operationTime': Timestamp(1771422537, 11)}, acknowledged=True)

In [26]:
from pymongo import UpdateOne

# Ta map de mots-clés
keywords_map = {
    "politique": ["manifestation", "coup d'État", "président", "gouvernement", "élection", "parlement", "ministre", "vote", "politique"],
    "économie": ["économie", "exportation", "industrie", "commerce", "finance", "banque", "bourse", "investissement", "entreprise", "marché"],
    "société": ["social", "communauté", "population", "culture", "démographie", "association"],
    "santé": ["santé", "médecin", "hôpital", "vaccin", "pandémie", "maladie", "traitement"],
    "éducation": ["éducation", "école", "université", "enseignement", "formation", "étudiant"],
    "technologie": ["technologie", "innovation", "science", "robotique", "informatique", "internet", "intelligence artificielle", "IA", "recherche"],
    "sport": ["football", "rugby", "match", "sport", "athlète", "compétition", "championnat"],
    "culture": ["art", "cinéma", "théâtre", "musique", "livre", "festival", "exposition"],
    "environnement": ["environnement", "climat", "écologie", "pollution", "déforestation", "biodiversité", "changement climatique"],
    "international": ["international", "ONU", "guerre", "conflit", "diplomatie", "pays", "monde", "relations internationales"],
    "justice": ["crime", "justice", "tribunal", "avocat", "police", "procès", "jugement"],
    "transport": ["transport", "route", "train", "aéroport", "trafic", "infrastructure", "mobilité"]
}

# Liste pour batch update
operations = []

# Parcourir tous les documents avec contenu
for doc in collection.find({"contenu": {"$type": "string"}}):
    texte = doc["contenu"].lower()
    categories = []
    
    # Vérifier chaque catégorie
    for cat, mots in keywords_map.items():
        if any(mot.lower() in texte for mot in mots):
            categories.append(cat)
    
    # Mise à jour si au moins une catégorie trouvée
    if categories:
        operations.append(
            UpdateOne(
                {"_id": doc["_id"]},
                {"$set": {"categorie": categories}}
            )
        )

# Exécution batch
if operations:
    result = collection.bulk_write(operations)
    print(f"Documents mis à jour avec catégories : {result.modified_count}")
else:
    print("Aucun document à mettre à jour")


Documents mis à jour avec catégories : 1647


In [31]:
df.head()

,_id,id_article,source,titre,date_publication,contenu,url,categorie,sentiment,source_type,created_at,sentiment_score
0,696f6450803a3bd1d5076f9f,https://www.madagascar-tribune.com/Conjoncture...,https://www.madagascar-tribune.com/spip.php?pa...,Richard Ravalomanana et Marie Michelle Sahondr...,2026-01-20 05:36:32,Deux figures emblématiques du régime Rajoelina...,https://www.madagascar-tribune.com/Conjoncture...,politique,neutre,NaN,NaN,NaN
1,696f6450803a3bd1d5076fa0,https://www.madagascar-tribune.com/Senat-la-HC...,https://www.madagascar-tribune.com/spip.php?pa...,Sénat : la HCC confirme la fin du mandat et l'...,2026-01-20 05:31:21,La Haute Cour constitutionnelle (HCC) a offici...,https://www.madagascar-tribune.com/Senat-la-HC...,autre,neutre,NaN,NaN,NaN
2,696f6451803a3bd1d5076fa1,https://www.madagascar-tribune.com/103-signatu...,https://www.madagascar-tribune.com/spip.php?pa...,103 poids plume,2026-01-19 05:35:00,Lors d'« Assises des Partis Politiques » pour ...,https://www.madagascar-tribune.com/103-signatu...,autre,negatif,NaN,NaN,NaN
3,696f6451803a3bd1d5076fa2,https://www.madagascar-tribune.com/Concertatio...,https://www.madagascar-tribune.com/spip.php?pa...,Concertation nationale : le président de la Re...,2026-01-19 05:00:00,Le colonel Randrianarina est venu à la rescous...,https://www.madagascar-tribune.com/Concertatio...,politique,positif,NaN,NaN,NaN
4,696f6451803a3bd1d5076fa3,https://www.madagascar-tribune.com/Administrat...,https://www.madagascar-tribune.com/spip.php?pa...,Herintsalama Rajaonarivelo ferme la porte à un...,2026-01-19 04:00:00,"Ce dimanche, devant des milliers de fidèles de...",https://www.madagascar-tribune.com/Administrat...,autre,neutre,NaN,NaN,NaN


In [35]:
from pymongo import UpdateOne

# Listes de référence
RSS_FEEDS = [
    "https://www.madagascar-tribune.com/spip.php?page=backend",
    "https://www.lexpress.mg/feeds/posts/default",
    "https://newsmada.com/feed/",
    "https://midi-madagasikara.mg/feed/",  
    "https://2424.mg/feed/",
    "https://rsf.org/fr/rss/afrique/madagascar/feed.xml",
    "https://www.madagate.org/index.php?format=feed&type=rss",
    "https://lgdi-madagascar.com/feed/",
    "https://midi-madagasikara.mg/category/politique/feed/",
    "https://midi-madagasikara.mg/category/economie/feed/",
    "https://www.lexpress.mg/feeds/posts/default/-/Politique",
    "https://www.lexpress.mg/feeds/posts/default/-/%C3%89conomie",
    "https://newsmada.com/category/les-nouvelles/feed/",
    "https://newsmada.com/category/les-nouvelles/politique/feed/",
    "https://2424.mg/category/actualite/politique/feed/",
    "https://2424.mg/category/actualite/economie/feed/",
    "https://www.lemonde.fr/madagascar/rss_full.xml",
    "https://www.courrierinternational.com/feed/rubrique/madagascar/rss.xml",
    "https://news.google.com/rss/search?q=Madagascar&hl=fr&gl=FR&ceid=FR:fr",
    "https://tanikomadagascar.com/feed/",
    "https://namana-studio.fr/feed/",
    "https://www.youtube.com/feeds/videos.xml?channel_id=UCK84qSI2bEMWkX9vUptkAlA"
]

SCRAP_HTML_SOURCES = ["Malagasy News"]
SCRAP_SELENIUM_SOURCES = ["Orange Actu"]

operations = []

# Parcourir uniquement les docs où source_type est vide ou None
for doc in collection.find({"$or": [{"source_type": {"$exists": False}}, {"source_type": None}, {"source_type": ""}]}):
    source = (doc.get("source") or "").strip()
    source_type = None

    # Vérifier RSS
    if any(source.startswith(rss) for rss in RSS_FEEDS):
        source_type = "rss"
    # Vérifier SCRAP_HTML
    elif any(source.startswith(scrap) for scrap in SCRAP_HTML_SOURCES):
        source_type = "scrap_html"
    # Vérifier SCRAP_SELENIUM
    elif any(scrap in source for scrap in SCRAP_SELENIUM_SOURCES):
        source_type = "scrap_selenium"
    # fallback
    else:
        source_type = "inconnu"

    operations.append(
        UpdateOne(
            {"_id": doc["_id"]},
            {"$set": {"source_type": source_type}}
        )
    )

# Exécution batch
if operations:
    result = collection.bulk_write(operations)
    print(f"Documents mis à jour avec source_type : {result.modified_count}")
else:
    print("Aucun document vide à mettre à jour")



Documents mis à jour avec source_type : 863


In [36]:
result = collection.aggregate([
    {"$group": {"_id": "$source_type", "count": {"$sum": 1}}}
])

for r in result:
    print(r)

{'_id': 'rss', 'count': 1900}
{'_id': 'scrap_html', 'count': 23}
{'_id': 'scrap_selenium', 'count': 68}


In [37]:
df.head()

,_id,id_article,source,titre,date_publication,contenu,url,categorie,sentiment,source_type,created_at,sentiment_score
0,696f6450803a3bd1d5076f9f,https://www.madagascar-tribune.com/Conjoncture...,https://www.madagascar-tribune.com/spip.php?pa...,Richard Ravalomanana et Marie Michelle Sahondr...,2026-01-20 05:36:32,Deux figures emblématiques du régime Rajoelina...,https://www.madagascar-tribune.com/Conjoncture...,politique,neutre,NaN,NaN,NaN
1,696f6450803a3bd1d5076fa0,https://www.madagascar-tribune.com/Senat-la-HC...,https://www.madagascar-tribune.com/spip.php?pa...,Sénat : la HCC confirme la fin du mandat et l'...,2026-01-20 05:31:21,La Haute Cour constitutionnelle (HCC) a offici...,https://www.madagascar-tribune.com/Senat-la-HC...,autre,neutre,NaN,NaN,NaN
2,696f6451803a3bd1d5076fa1,https://www.madagascar-tribune.com/103-signatu...,https://www.madagascar-tribune.com/spip.php?pa...,103 poids plume,2026-01-19 05:35:00,Lors d'« Assises des Partis Politiques » pour ...,https://www.madagascar-tribune.com/103-signatu...,autre,negatif,NaN,NaN,NaN
3,696f6451803a3bd1d5076fa2,https://www.madagascar-tribune.com/Concertatio...,https://www.madagascar-tribune.com/spip.php?pa...,Concertation nationale : le président de la Re...,2026-01-19 05:00:00,Le colonel Randrianarina est venu à la rescous...,https://www.madagascar-tribune.com/Concertatio...,politique,positif,NaN,NaN,NaN
4,696f6451803a3bd1d5076fa3,https://www.madagascar-tribune.com/Administrat...,https://www.madagascar-tribune.com/spip.php?pa...,Herintsalama Rajaonarivelo ferme la porte à un...,2026-01-19 04:00:00,"Ce dimanche, devant des milliers de fidèles de...",https://www.madagascar-tribune.com/Administrat...,autre,neutre,NaN,NaN,NaN


In [44]:
from pymongo import MongoClient
import pandas as pd
client = MongoClient(MONGO_URI)
db = client["veille_media"]
collection = db["articles"]
data = list(collection.find())
df = pd.DataFrame(data)

df.head()

,_id,id_article,source,titre,date_publication,contenu,url,categorie,sentiment,source_type,created_at,sentiment_score
0,696f6450803a3bd1d5076f9f,https://www.madagascar-tribune.com/Conjoncture...,https://www.madagascar-tribune.com/spip.php?pa...,Richard Ravalomanana et Marie Michelle Sahondr...,2026-01-20 05:36:32,Deux figures emblématiques du régime Rajoelina...,https://www.madagascar-tribune.com/Conjoncture...,"[politique, éducation, technologie]",neutre,rss,2026-01-20 05:36:32,0.0000
1,696f6450803a3bd1d5076fa0,https://www.madagascar-tribune.com/Senat-la-HC...,https://www.madagascar-tribune.com/spip.php?pa...,Sénat : la HCC confirme la fin du mandat et l'...,2026-01-20 05:31:21,La Haute Cour constitutionnelle (HCC) a offici...,https://www.madagascar-tribune.com/Senat-la-HC...,[politique],neutre,rss,2026-01-20 05:31:21,-0.0125
2,696f6451803a3bd1d5076fa1,https://www.madagascar-tribune.com/103-signatu...,https://www.madagascar-tribune.com/spip.php?pa...,103 poids plume,2026-01-19 05:35:00,Lors d'« Assises des Partis Politiques » pour ...,https://www.madagascar-tribune.com/103-signatu...,"[politique, technologie, culture]",negatif,rss,2026-01-19 05:35:00,-0.4000
3,696f6451803a3bd1d5076fa2,https://www.madagascar-tribune.com/Concertatio...,https://www.madagascar-tribune.com/spip.php?pa...,Concertation nationale : le président de la Re...,2026-01-19 05:00:00,Le colonel Randrianarina est venu à la rescous...,https://www.madagascar-tribune.com/Concertatio...,"[politique, technologie, culture]",positif,rss,2026-01-19 05:00:00,0.2000
4,696f6451803a3bd1d5076fa3,https://www.madagascar-tribune.com/Administrat...,https://www.madagascar-tribune.com/spip.php?pa...,Herintsalama Rajaonarivelo ferme la porte à un...,2026-01-19 04:00:00,"Ce dimanche, devant des milliers de fidèles de...",https://www.madagascar-tribune.com/Administrat...,"[politique, culture]",neutre,rss,2026-01-19 04:00:00,0.0000


In [39]:
from pymongo import UpdateOne

operations = []

# Parcourir uniquement les documents où created_at est vide ou None
for doc in collection.find({"$or": [{"created_at": {"$exists": False}}, {"created_at": None}, {"created_at": ""}]}):
    date_pub = doc.get("date_publication")
    
    if date_pub:  # on ne copie que si date_publication existe
        operations.append(
            UpdateOne(
                {"_id": doc["_id"]},
                {"$set": {"created_at": date_pub}}
            )
        )

# Exécution batch
if operations:
    result = collection.bulk_write(operations)
    print(f"Documents mis à jour avec created_at : {result.modified_count}")
else:
    print("Aucun document vide à mettre à jour")


Documents mis à jour avec created_at : 874


In [40]:
result = collection.aggregate([
     {"$group": {"_id": "$created_at", "count": {"$sum": 1}}}
])

for r in result:
    print(r)

{'_id': '2026-02-11T12:37:25.137031', 'count': 1}
{'_id': datetime.datetime(2026, 1, 12, 6, 40, 50), 'count': 1}
{'_id': datetime.datetime(2026, 1, 16, 6, 32, 53), 'count': 1}
{'_id': datetime.datetime(2026, 1, 6, 13, 54, 58), 'count': 1}
{'_id': '2026-02-10T14:24:10.067032', 'count': 1}
{'_id': datetime.datetime(2026, 1, 3, 8, 0), 'count': 3}
{'_id': '2026-02-10T14:06:10.468266', 'count': 1}
{'_id': '2026-02-13T08:43:34.006679', 'count': 1}
{'_id': '2026-02-16T15:12:52.611951', 'count': 1}
{'_id': '2026-02-17T08:35:33.615506', 'count': 1}
{'_id': '2026-02-11T12:38:26.611486', 'count': 1}
{'_id': '2026-02-10T16:02:10.274977+00:00', 'count': 1}
{'_id': '2026-02-18T09:57:53.909447', 'count': 1}
{'_id': '2026-02-10T16:02:06.098916+00:00', 'count': 1}
{'_id': '2026-02-18T09:57:44.607876', 'count': 1}
{'_id': datetime.datetime(2026, 1, 21, 4, 40), 'count': 1}
{'_id': datetime.datetime(2026, 1, 17, 13, 3, 20), 'count': 1}
{'_id': '2026-02-12T16:31:14.555231', 'count': 1}
{'_id': datetime.dat

In [43]:
import sys
import os
from pymongo import UpdateOne

# Ajouter le dossier parent où se trouve transform.py
sys.path.append(r"D:/Henintsoa_/WEB DESIGN OUTILS/veille_media_mada/etl")

# Importer la fonction
from transform import clean_text, analyze_sentiment_score

# Créer une fonction qui calcule sentiment_score numérique pour les docs existants
def update_sentiment_score_numeric(collection):
    """
    Calcule le score de sentiment (-1 à +1) pour tous les documents
    où sentiment_score est vide ou None, et met à jour la base.
    """
    operations = []

    # Sélectionner uniquement les documents où sentiment_score est vide
    for doc in collection.find({"$or": [{"sentiment_score": {"$exists": False}},
                                        {"sentiment_score": None},
                                        {"sentiment_score": ""}]}):
        texte = doc.get("contenu", "")
        texte_clean = clean_text(texte)  # Nettoyage HTML
        score = analyze_sentiment_score(texte_clean)  # Score numérique

        operations.append(
            UpdateOne(
                {"_id": doc["_id"]},
                {"$set": {"sentiment_score": score}}
            )
        )

    # Mise à jour en batch
    if operations:
        result = collection.bulk_write(operations)
        print(f"Documents mis à jour avec sentiment_score numérique : {result.modified_count}")
    else:
        print("Aucun document vide à mettre à jour pour sentiment_score")

# Exemple d'utilisation
update_sentiment_score_numeric(collection)


Documents mis à jour avec sentiment_score numérique : 1991


In [45]:
# Vérifier le nombre de documents par sentiment_score
result = collection.aggregate([
    {"$group": {"_id": "$sentiment_score", "count": {"$sum": 1}}},
    {"$sort": {"count": -1}}
])

for r in result:
    print(r)


{'_id': 0.0, 'count': 1487}
{'_id': 0.2, 'count': 44}
{'_id': 0.1, 'count': 42}
{'_id': 0.5, 'count': 33}
{'_id': 0.4, 'count': 15}
{'_id': 0.16666666666666666, 'count': 13}
{'_id': 0.375, 'count': 11}
{'_id': -0.9, 'count': 10}
{'_id': -0.0125, 'count': 10}
{'_id': -0.8, 'count': 9}
{'_id': 0.05, 'count': 9}
{'_id': 0.21428571428571427, 'count': 8}
{'_id': 0.25, 'count': 8}
{'_id': -0.4, 'count': 7}
{'_id': 0.3, 'count': 7}
{'_id': -0.13333333333333333, 'count': 6}
{'_id': -0.05, 'count': 6}
{'_id': -0.06666666666666667, 'count': 6}
{'_id': 0.03333333333333333, 'count': 6}
{'_id': 0.13333333333333333, 'count': 6}
{'_id': -0.00625, 'count': 6}
{'_id': 0.10714285714285714, 'count': 5}
{'_id': -0.5, 'count': 5}
{'_id': 0.15, 'count': 5}
{'_id': 0.06666666666666667, 'count': 5}
{'_id': 0.225, 'count': 4}
{'_id': 0.125, 'count': 4}
{'_id': -0.125, 'count': 4}
{'_id': -0.025, 'count': 4}
{'_id': 0.016666666666666666, 'count': 4}
{'_id': -0.1, 'count': 4}
{'_id': 0.275, 'count': 3}
{'_id': 0